# Phase 2 — Feature Check
ตรวจสอบ features.csv: shape, NaN, distribution แยก Day

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('../data/features.csv')
print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
df.head(3)

In [ ]:
print('NaN count:')
print(df.isna().sum()[df.isna().sum() > 0])
print(f'\nDay range: {df["day"].min()} - {df["day"].max()}')
print(df.groupby(['variety','day']).size().unstack(fill_value=0))

## Key color features แยก Day

In [ ]:
features_to_plot = ['pct_green', 'pct_brown', 'L_mean', 'a_mean']
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, feat in zip(axes.flat, features_to_plot):
    for var, grp in df.groupby('variety'):
        means = grp.groupby('day')[feat].mean()
        ax.plot(means.index, means.values, marker='o', label=var)
    ax.set_title(feat)
    ax.set_xlabel('Day')
    ax.legend()
plt.suptitle('Mean features by Day', fontsize=13)
plt.tight_layout()
plt.savefig('../results/feature_trends.png', dpi=80, bbox_inches='tight')
plt.show()

## Correlation heatmap

In [ ]:
feat_cols = ['L_mean','L_std','a_mean','a_std','b_mean','b_std',
             'pct_green','pct_yellow','pct_brown',
             'contrast','correlation','energy','homogeneity',
             'area_ratio','day']
corr = df[feat_cols].corr()
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            linewidths=0.5, ax=ax, annot_kws={'size': 7})
ax.set_title('Feature correlation (incl. day)')
plt.tight_layout()
plt.savefig('../results/feature_corr.png', dpi=80, bbox_inches='tight')
plt.show()

## สรุปผลตรวจ

In [ ]:
review = {
    'rows': len(df),
    'nan_temp_rows': int(df['temp_min'].isna().sum()),
    'top_features': ['a_mean (0.86)', 'area_ratio (-0.74)', 'pct_yellow (0.72)', 'pct_green (-0.68)'],
    'weak_features': ['L_mean (0.27 — non-linear)', 'energy (~0)', 'homogeneity (~0)'],
    'notes': (
        'L_mean ลงช่วง D0-D4 แล้วขึ้นตอนใบน้ำตาลจัด (non-linear) — '
        'keep ไว้ให้ model จัดการ; '
        'energy/homogeneity corr~0 — ตัดออกตอน Phase 4; '
        'a_mean เป็น best single predictor'
    ),
}
for k, v in review.items():
    print(f'{k}: {v}')